# P3, Joining data

---

In this part we will be collecting the stock data that will match up with each row of the dataframe being collecting, and then joining that data with the dataframe.

--- 
## The data we want:
For this dataset I have decided that I want the daily high stock sprice of each transaction with each of these dates relative to the trade:
- The price of the stock on the date of purchase (`dt`)
- The price of the stock on the date of notification (`ndt`)
- The price of the stock on the date of the publishing of the PTR 
    - This is another data join that we will be doing!
- The price of the stock 45 days after the purchase
- The price of the stock today (10/20/26 will be "today")

## Creating and caching all of the stock data we need.

In order to get all of the historical stock data in the most polite way possible, we will:
- assemble our dataframe with all of our rows
- collect all of the desired stocks and all of days in which we want the daily high
- Find and remove duplicates. If we have multiple transactions that purchased the same stock on the same day, then we only need one request!
- Export to a json that we can read to build the data cache
- Use the `yfiannce` scraper package to scrape the stock data over a long period of time, building a cache of all the data we need
- Join the stock data to our dataframe to have the stock data that we want


## Step 1: Reassembling our dataframe

In [1]:
import pandas as pd


base_url = "./../data/json/"
index_url = "./../data/all_json_paths.txt"


document_ids = (
    pd.read_csv(index_url, header=None, dtype=str)[0]
    .str.strip()
    .dropna()
)
document_ids = document_ids[document_ids != ""].tolist()

json_files = [f"{document_id}.json" for document_id in document_ids]
dataframes = []

for file_name in json_files:
    df = pd.read_json(f"{base_url}{file_name}")
    df["file_name"] = file_name
    dataframes.append(df)

combined_df = (
    pd.concat(dataframes, ignore_index=True)
    if dataframes
    else pd.DataFrame()
)

transaction_columns = pd.json_normalize(
    combined_df["transactions"].map(
        lambda value: value if isinstance(value, dict) else {}
    ),
    sep="_"
)

parsed_df = pd.concat(
    [
        combined_df.drop(columns="transactions").reset_index(drop=True),
        transaction_columns.reset_index(drop=True),
    ],
    axis=1
)


# Making a ticker column by searching for a regex pattern between parentheses in the 'ast' column
parsed_df["ticker"] = parsed_df["ast"].str.extract(r"\((.*?)\)")[0]

# removing values in the 'ticker' column that are longer than 5 characters
parsed_df.loc[parsed_df["ticker"].str.len() > 5, "ticker"] = None
# This removes the values of 128 rows

# Fixing date columns to datetime format
for column in ["dt", "ndt"]:
    parsed_df[column] = pd.to_datetime(
        parsed_df[column],
        format="%m/%d/%Y",
        errors="coerce"
    )

# Theres apparently a lot of malformed sentinel dates that make no sense, so we will filter them out
lower_bound = pd.Timestamp("2022-11-14")
upper_bound = pd.Timestamp("2026-01-01")

# Classifying each row on whether or not it appeared in a handwritten PTR doc or a digital PTR doc
parsed_df["handwritten"] = parsed_df["file_name"].str.extract(r"(\d+)\.json")[0].astype(int) < 20000000


for column in ["dt", "ndt"]:
    outside_range = (
        (parsed_df[column] < lower_bound)
        | (parsed_df[column] > upper_bound)
    )
    parsed_df.loc[outside_range, column] = pd.NaT
print(parsed_df.shape)

print("Handwritten row count:")
print(parsed_df["handwritten"].sum())
print("Digital row count:")
print((~parsed_df["handwritten"]).sum())

parsed_df.head(5)

(29936, 14)
Handwritten row count:
16870
Digital row count:
13066


,file_name,own,ast,code,typ,dt,ndt,amt,cg,stat,sub,desc,ticker,handwritten
0,20022986.json,,"Activision Blizzard, Inc (ATVI)",ST,P,2023-04-20,2023-05-05,"$1,001 - $15,000",False,New,Charles Schwab IRA,,ATVI,False
1,20022986.json,SP,SouthState Corporation (SSB),ST,S,2023-03-27,2023-04-05,"$100,001 - $250,000",True,New,"R.W. Allen & Associates, Inc.",,SSB,False
2,20022986.json,SP,The Charles Schwab Corporation Depositary Shar...,ST,P,2023-05-18,2023-05-17,"$15,001 - $50,000",False,New,"R.W. Allen & Associates, Inc.",,NaN,False
3,20023082.json,SP,3M Company (MMM),ST,S,NaT,NaT,"$15,001 - $50,000",False,Amended,"R.W. Allen & Associates, Inc. > RWA&A - Securi...",,MMM,False
4,20023082.json,SP,3M Company (MMM),ST,S,NaT,NaT,"$1,001 - $15,000",False,New,LIVTR,,MMM,False


# Part 2: The bonus join to match the documents with their congressmen/women

In [2]:
ptr_df = pd.read_csv(
    "./../data/clerk_directories/combined_ptr.csv",
    dtype=str,
    engine="python",
    on_bad_lines="warn",
)

lookup = ptr_df.iloc[:, [0, 1, 3, -2, -1]].copy()
lookup.columns = ["last", "first", "StateDst", "pdt", "DocID"]
lookup["DocID"] = lookup["DocID"].str.strip()
lookup["pdt"] = pd.to_datetime(
    pd.to_numeric(lookup["pdt"], errors="coerce"),
    unit="D",
    origin="1899-12-30",
)
lookup = lookup.drop_duplicates("DocID", keep="first")

parsed_df["DocID"] = parsed_df["file_name"].str.replace(r"\.json$", "", regex=True)
parsed_df = parsed_df.merge(
    lookup,
    on="DocID",
    how="left",
    validate="many_to_one",
).drop(columns="DocID")

parsed_df.head()


,file_name,own,ast,code,typ,dt,ndt,amt,cg,stat,sub,desc,ticker,handwritten,last,first,StateDst,pdt
0,20022986.json,,"Activision Blizzard, Inc (ATVI)",ST,P,2023-04-20,2023-05-05,"$1,001 - $15,000",False,New,Charles Schwab IRA,,ATVI,False,Allen,Richard W.,GA12,2023-06-02
1,20022986.json,SP,SouthState Corporation (SSB),ST,S,2023-03-27,2023-04-05,"$100,001 - $250,000",True,New,"R.W. Allen & Associates, Inc.",,SSB,False,Allen,Richard W.,GA12,2023-06-02
2,20022986.json,SP,The Charles Schwab Corporation Depositary Shar...,ST,P,2023-05-18,2023-05-17,"$15,001 - $50,000",False,New,"R.W. Allen & Associates, Inc.",,NaN,False,Allen,Richard W.,GA12,2023-06-02
3,20023082.json,SP,3M Company (MMM),ST,S,NaT,NaT,"$15,001 - $50,000",False,Amended,"R.W. Allen & Associates, Inc. > RWA&A - Securi...",,MMM,False,Allen,Richard W.,GA12,2023-08-10
4,20023082.json,SP,3M Company (MMM),ST,S,NaT,NaT,"$1,001 - $15,000",False,New,LIVTR,,MMM,False,Allen,Richard W.,GA12,2023-08-10


In [3]:
# Assert that the merge was successful
assert parsed_df["last"].notna().all(), "Some DocIDs did not match in the lookup table"
assert parsed_df["first"].notna().all(), "Some DocIDs did not match in the lookup table"
assert parsed_df["StateDst"].notna().all(), "Some DocIDs did not match in the lookup table"

# Part 3: Culling the data to just what we can analyze
We want to scrape and cache as little data as possible, which means we need to get rid of all of the data we cannot completely analyze, so that we don't have to scrape the values for it.

For the data to be something that we can analyze, all of the conditions must be met:

- It must have a stock/ETF ticker
    - While insider trading via an ETF is highly unlikely, I am still going to check the results for fun.
- It must have both date values
- It must be a `New` status

In [4]:
culled_df = parsed_df.loc[
    parsed_df["ticker"].notna()
    & parsed_df["stat"].eq("New")
    & parsed_df["dt"].notna()
    & parsed_df["ndt"].notna()
].copy()

print(parsed_df.shape)
print(culled_df.shape)
culled_df.head()

# See how many rows were culled, grouped by 'handwritten'
culled_counts = parsed_df.shape[0] - culled_df.shape[0]
print("Total rows culled:", culled_counts)

print("rows preserved by 'handwritten':")
print(culled_df["handwritten"].value_counts())

(29936, 18)
(11376, 18)
Total rows culled: 18560
rows preserved by 'handwritten':
handwritten
False    10917
True       459
Name: count, dtype: int64


Wow, the LLM did not get to parse all of the neccesary data very often.

---

# Part 4: Generating the list of stock data we need:

With our remaining data, we will make a list of the stock tickers and dates for those stock tickers that we want to cache, and then we will use the scraping program `yfiannce_cache.py` in order to build a cache of all of the stock data.

To re-iterate, we are looking for the daily high stock price of each stock on each of these dates relative to the trade:
- The price of the stock on the date of purchase (`dt`)
- The price of the stock on the date of notification (`ndt`)
- The price of the stock on the date of the publishing of the PTR 
    - This is another data join that we will be doing!
- The price of the stock 45 days after the purchase (The day they are legally required to report the trade by)
- The price of the stock today (10/20/26 will be "today")
 

In [5]:
today = pd.Timestamp("2026-10-20")

date_requests = pd.concat(
    [
        culled_df[["ticker", "dt"]].rename(columns={"dt": "date"}).assign(event="purchase"),
        culled_df[["ticker", "ndt"]].rename(columns={"ndt": "date"}).assign(event="notification"),
        culled_df[["ticker", "pdt"]].rename(columns={"pdt": "date"}).assign(event="PTR publication"),
        culled_df[["ticker", "dt"]]
        .assign(date=lambda df: df["dt"] + pd.Timedelta(days=45))
        [["ticker", "date"]]
        .assign(event="45 days after purchase"),
        culled_df[["ticker"]].assign(date=today, event="today"),
    ],
    ignore_index=True,
)

date_requests["date"] = pd.to_datetime(date_requests["date"]).dt.normalize()
date_requests = date_requests.dropna(subset=["ticker", "date"])

# Unique ticker/date pairs to fetch; date_requests retains the event labels.
scrape_dates = (
    date_requests[["ticker", "date"]]
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)
print("scrape dates before dropping duplicates")
pre_dropped_len = len(scrape_dates)
print(pre_dropped_len)
scrape_dates = scrape_dates.drop_duplicates()

print(f"Unique ticker/date pairs: {len(scrape_dates):,}")
print(f"Dropped {pre_dropped_len - len(scrape_dates):,} duplicate rows")
print("Dates to scrape by ticker:")

scrape_dates.groupby("ticker").size().sort_values(ascending=False)
# Save the scrape_dates DataFrame to a CSV file
scrape_dates.to_csv("./../data/scrape_dates.csv", index=False)


scrape dates before dropping duplicates
56880
Unique ticker/date pairs: 36,283
Dropped 20,597 duplicate rows
Dates to scrape by ticker:
